In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gdown
import os
import re

# --- 1. SETUP & DIRECTORY STRUCTURE ---
os.makedirs('csv_files', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

def get_direct_link(url):
    file_id_match = re.search(r'd/([a-zA-Z0-9_-]+)', url)
    if not file_id_match:
        file_id_match = re.search(r'id=([a-zA-Z0-9_-]+)', url)
    if file_id_match:
        return f'https://drive.google.com/uc?id={file_id_match.group(1)}'
    return url

raw_trader_link = 'https://drive.google.com/file/d/1IAfLZwu6rJzyWKgBToqwSmmVYU6VbjVs/view?usp=sharing'
raw_sentiment_link = 'https://drive.google.com/file/d/1PgQC0tO8XN-wqkNyghWc_-mnrYv_nhSf/view?usp=sharing'

trader_url = get_direct_link(raw_trader_link)
sentiment_url = get_direct_link(raw_sentiment_link)

print("Step 1: Downloading datasets...")
gdown.download(trader_url, 'csv_files/hyperliquid_trader_data.csv', quiet=False)
gdown.download(sentiment_url, 'csv_files/fear_greed_index.csv', quiet=False)

# --- 2. DATA LOADING & CLEANING ---

def clean_df(df):
    # Strip whitespace and quotes from headers
    df.columns = [str(c).replace("'", "").replace('"', "").strip() for c in df.columns]
    # Strip whitespace from string columns
    for col in df.select_dtypes(include=['object']):
        df[col] = df[col].astype(str).str.strip()
    return df

def map_columns(df):
    """Smart mapping for column names based on keywords."""
    cols = df.columns
    rename_map = {}

    # Define possible variations for key columns
    mappings = {
        'side': ['side', 'direction', 'buy/sell'],
        'leverage': ['leverage', 'lev', 'margin'],
        'closedPnL': ['closedpnl', 'pnl', 'profit', 'realized', 'p&l'],
        'size': ['size tokens', 'size', 'amount', 'quantity', 'qty'],
        'size_usd': ['size usd', 'position value', 'notional']
    }

    for standard_name, aliases in mappings.items():
        # Skip if already exists
        if standard_name in cols:
            continue

        # Check aliases
        for alias in aliases:
            match = next((c for c in cols if alias in c.lower()), None)
            if match:
                rename_map[match] = standard_name
                break

    if rename_map:
        print(f"Mapping columns: {rename_map}")
        df = df.rename(columns=rename_map)

    return df

try:
    print("\nStep 2: Loading and cleaning data...")
    # --- SENTIMENT DATA ---
    df_sentiment = pd.read_csv('csv_files/fear_greed_index.csv')
    df_sentiment = clean_df(df_sentiment)

    sent_date_col = next((c for c in df_sentiment.columns if 'date' in c.lower()), df_sentiment.columns[0])
    df_sentiment['Date'] = pd.to_datetime(df_sentiment[sent_date_col], format='mixed', dayfirst=True)

    class_col = next((c for c in df_sentiment.columns if 'classification' in c.lower()), None)
    if not class_col:
        raise KeyError(f"Classification column not found. Available: {list(df_sentiment.columns)}")
    df_sentiment['Classification'] = df_sentiment[class_col]

    # --- TRADER DATA ---
    df_trader = pd.read_csv('csv_files/hyperliquid_trader_data.csv')
    df_trader = clean_df(df_trader)

    print(f"Raw Trader Columns: {list(df_trader.columns)}") # DEBUG PRINT

    # Map columns like 'Realized PnL' -> 'closedPnL'
    df_trader = map_columns(df_trader)

    # Verify critical columns
    required = ['side', 'closedPnL']
    missing = [c for c in required if c not in df_trader.columns]
    if missing:
        print(f"WARNING: Missing columns {missing}. Analysis may be incomplete.")

    # Time Column Logic
    trader_time_col = next((c for c in df_trader.columns if 'time' in c.lower() or 'date' in c.lower()), None)

    if trader_time_col:
        print(f"Using '{trader_time_col}' as time column.")
        # Attempt mixed format first
        df_trader['time_dt'] = pd.to_datetime(df_trader[trader_time_col], format='mixed', dayfirst=True, errors='coerce')
        # Fallback to unix timestamp if strings failed
        nan_mask = df_trader['time_dt'].isna()
        if nan_mask.any():
             # Check if it looks like seconds (10 digits) or ms (13 digits)
            first_val = df_trader.loc[nan_mask, trader_time_col].iloc[0]
            unit = 's' if len(str(first_val)) <= 10 else 'ms'
            df_trader.loc[nan_mask, 'time_dt'] = pd.to_datetime(df_trader.loc[nan_mask, trader_time_col], unit=unit, errors='coerce')

        df_trader['Date'] = df_trader['time_dt'].dt.normalize()
    else:
        raise KeyError("Required time/date column missing from trader data.")

    # --- MERGE ---
    df_trader = df_trader.dropna(subset=['Date'])
    df_sentiment = df_sentiment.dropna(subset=['Date'])
    df_merged = pd.merge(df_trader, df_sentiment[['Date', 'Classification']], on='Date', how='inner')

except Exception as e:
    print(f"FAILED: {e}")
    import traceback
    traceback.print_exc()
    df_merged = pd.DataFrame()

# --- 3. ANALYSIS & VISUALIZATION ---

if not df_merged.empty:
    print(f"\nStep 3: Analyzing {len(df_merged)} overlapping records...")

    # Strategy Definition
    def label_strategy(row):
        cls = str(row['Classification']).lower()
        side = str(row['side']).upper().strip() if 'side' in row else 'UNKNOWN'

        if 'fear' in cls and side in ['B', 'LONG', 'BUY']: return 'Contrarian (Long on Fear)'
        if 'greed' in cls and side in ['S', 'SHORT', 'SELL']: return 'Contrarian (Short on Greed)'
        return 'Trend Following'

    df_merged['Strategy'] = df_merged.apply(label_strategy, axis=1)

    # Output 1: Risk Analysis (Leverage OR Size USD)
    plt.figure(figsize=(10, 6))

    if 'leverage' in df_merged.columns:
        # Convert to numeric, forcing errors to NaN
        df_merged['leverage'] = pd.to_numeric(df_merged['leverage'], errors='coerce')
        # Fix: Added hue=Classification and legend=False to fix warning
        sns.barplot(data=df_merged, x='Classification', y='leverage', hue='Classification', palette='coolwarm', errorbar=None, legend=False)
        plt.title('Average Leverage Used per Market Sentiment')
        plt.savefig('outputs/leverage_vs_sentiment.png')
        print("Generated: outputs/leverage_vs_sentiment.png")

    elif 'size_usd' in df_merged.columns:
        # FALLBACK: Use Size USD as a proxy for conviction/risk
        df_merged['size_usd'] = pd.to_numeric(df_merged['size_usd'], errors='coerce')
        # Fix: Added hue=Classification and legend=False to fix warning
        sns.barplot(data=df_merged, x='Classification', y='size_usd', hue='Classification', palette='coolwarm', errorbar=None, legend=False)
        plt.title('Average Trade Size (USD) per Market Sentiment')
        plt.ylabel('Size (USD)')
        plt.savefig('outputs/size_usd_vs_sentiment.png')
        print("Generated: outputs/size_usd_vs_sentiment.png (Fallback for missing Leverage)")

    else:
        print("Skipping Risk plot (neither 'leverage' nor 'size_usd' found)")
    plt.close()

    # Output 2: Profitability Boxplot
    plt.figure(figsize=(12, 6))
    if 'closedPnL' in df_merged.columns:
        # Clean PnL data (remove 'USDT' symbols if present)
        if df_merged['closedPnL'].dtype == object:
             df_merged['closedPnL'] = df_merged['closedPnL'].astype(str).str.replace('USDT', '').str.replace(',', '')
        df_merged['closedPnL'] = pd.to_numeric(df_merged['closedPnL'], errors='coerce')

        # Fix: Added hue=Strategy and legend=False to fix warning
        sns.boxplot(data=df_merged, x='Strategy', y='closedPnL', hue='Strategy', showfliers=False, legend=False)
        plt.axhline(0, color='red', linestyle='--', alpha=0.5)
        plt.title('PnL Distribution: Strategy Performance')
        plt.savefig('outputs/strategy_performance.png')
        print("Generated: outputs/strategy_performance.png")

        # Final Summary
        print("\n--- RESULTS SUMMARY ---")
        print(df_merged.groupby('Strategy')['closedPnL'].agg(['count', 'mean', 'median']))
    else:
        print("Skipping PnL plot (column missing)")
    plt.close()

    # Save processed data
    df_merged.to_csv('csv_files/processed_analysis_data.csv', index=False)
    print("\nSUCCESS: Analysis complete.")

else:
    print("\nERROR: No overlapping dates found or data loading failed.")

Step 1: Downloading datasets...


Downloading...
From: https://drive.google.com/uc?id=1IAfLZwu6rJzyWKgBToqwSmmVYU6VbjVs
To: /content/csv_files/hyperliquid_trader_data.csv
100%|██████████| 47.5M/47.5M [00:01<00:00, 35.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PgQC0tO8XN-wqkNyghWc_-mnrYv_nhSf
To: /content/csv_files/fear_greed_index.csv
100%|██████████| 90.8k/90.8k [00:00<00:00, 58.0MB/s]



Step 2: Loading and cleaning data...
Raw Trader Columns: ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']
Mapping columns: {'Side': 'side', 'Closed PnL': 'closedPnL', 'Size Tokens': 'size', 'Size USD': 'size_usd'}
Using 'Timestamp IST' as time column.

Step 3: Analyzing 211218 overlapping records...
Generated: outputs/size_usd_vs_sentiment.png (Fallback for missing Leverage)
Generated: outputs/strategy_performance.png

--- RESULTS SUMMARY ---
                              count       mean    median
Strategy                                                
Contrarian (Long on Fear)     41205  56.015456  0.000000
Contrarian (Short on Greed)   47779  85.026753  0.076594
Trend Following              122234  31.774112  0.000000

SUCCESS: Analysis complete.
